In [3]:
import pandas as pd
import xarray as xr
ds_big = xr.open_zarr("/Users/abhimanyu/Downloads/IFS_reforecast_download-main/s2s_new_vars_sorted.zarr")
ds_imd = xr.open_zarr("../data/raw/IMD_rainfall_0p25.zarr")
ds_imd = ds_imd.where(ds_imd != -999)
ds_ecm = xr.open_zarr("../data/processed/s2s_reforecast_sorted.zarr")

# convert lead days into timedeltas
step_td = pd.to_timedelta(ds_ecm.step.values, unit="D").to_numpy()

ds_ecmv = ds_ecm.assign_coords(
    valid_time=(("time", "step"),
                ds_ecm.time.values[:, None] + step_td[None, :])
)

In [13]:
# ======================================================================
# CLEAN DUAL-ENCODER UNet -- Streamlined and Scientifically Sound
#
# CHANGES FROM PREVIOUS PIPELINE:
# 1. REMOVED TIME LAGS: The 6x channel bloat was causing immediate overfitting
#    and feeding the model stale 4-week-old forecasts.
# 2. RAW CLIMATOLOGY INCLUDED: The model now explicitly receives the DOY
#    climatology as a feature, not just anomalies, so it knows the base state.
# 3. CLEAN ANOMALISATION: Simplified the normalisation logic so it's readable.
# 4. MEMORY OPTIMISED: Uses a single clean DataGenerator rather than complex
#    memory-mapped arrays that break multiprocessing.
# ======================================================================

import os, json, time, gc
from contextlib import contextmanager
import numpy as np
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# ---------------- CONFIG ----------------
IMD_TARGET_VAR = "rain"
WINDOWS = [("week2", 8, 14), ("week3_4", 15, 28), ("week5_6", 29, 42)]
CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0
TEST_YEARS_N = 3
DTYPE = np.float32

BIG_VARS = [
    "top_net_thermal_radiation", 
    "geopotential_height_200",
    "geopotential_height_500", 
    "geopotential_height_850",
    "geopotential_height_1000"
]

CACHE_DIR = "../data/cache/unet_cache_clean"
OUT_MAPS = "../results/models/unet_clean_pred.nc"

class Args:
    cmd = "prepare"
    epochs = 40          # Lowered, as cleaner data needs fewer epochs
    batch = 16           # Back up to 16 since we removed the lag bloat
    base = 32            # Slightly wider network for spatial learning
    drop = 0.3           # Increased dropout to fight overfitting
    wd = 1e-3
    lr = 2e-4
    patience = 8
    folds = 5

args = Args()

@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)

# ======================================================================
# 1. PREPARATION & CACHING (Cleaned)
# ======================================================================

def _subset_box(ds, imd, pad):
    """Spatially crop ECMWF to match IMD target."""
    for c in ("lat", "lon"):
        if ds[c].values[0] > ds[c].values[-1]:
            ds = ds.sortby(c)
    if pad is not None:
        la0, la1 = float(imd.lat.min()), float(imd.lat.max())
        lo0, lo1 = float(imd.lon.min()), float(imd.lon.max())
        ds = ds.sel(lat=slice(la0 - pad, la1 + pad), lon=slice(lo0 - pad, lo1 + pad))
    return ds

def _window_mean(sub, feature_vars, leads, lo, hi):
    """Average the daily steps into a single window block."""
    sel = np.where((leads >= lo) & (leads <= hi))[0]
    if len(sel) == 0: raise ValueError(f"No leads in [{lo},{hi}]")
    Xw = sub.isel(step=sel).mean(dim="step").compute()
    return np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE), sel

def prepare_clean(ecmwf_ds, big_ds, imd_ds, cache_path):
    os.makedirs(cache_path, exist_ok=True)
    imd = imd_ds[IMD_TARGET_VAR].assign_coords(time=imd_ds["time"].dt.floor("D"))
    
    subA = _subset_box(ecmwf_ds, imd, COARSE_PAD)
    subB = _subset_box(big_ds, imd, pad=None)
    
    initA = subA["time"].values
    leadsA = (subA["step"].values / np.timedelta64(1, "D")).astype(int)
    leadsB = (subB["step"].values / np.timedelta64(1, "D")).astype(int)
    
    XA_list, XB_list, y_list, doy_list, wid_list = [], [], [], [], []
    
    for wi, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Extracting {wname} (days {lo}-{hi})"):
            xa, sel = _window_mean(subA, list(subA.data_vars), leadsA, lo, hi)
            xb, _ = _window_mean(subB, BIG_VARS, leadsB, lo, hi)
            
            vt = subA["valid_time"].isel(step=sel).dt.floor("D").values
            y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            y_mean = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)
            
            centre = initA + np.timedelta64((lo + hi) // 2, "D")
            doy = xr.DataArray(centre, dims="t").dt.dayofyear.values
            
            XA_list.append(xa)
            XB_list.append(xb)
            y_list.append(y_mean)
            doy_list.append(doy)
            wid_list.append(np.full(len(initA), wi, np.int64))
            
    XA = np.concatenate(XA_list, axis=0)
    XB = np.concatenate(XB_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    doy = np.concatenate(doy_list, axis=0)
    wid = np.concatenate(wid_list, axis=0)
    year = np.tile(initA.astype("datetime64[Y]").astype(int) + 1970, len(WINDOWS))
    mask = np.isfinite(y).all(axis=0)

    # Save to disk as clean numpy arrays
    np.save(os.path.join(cache_path, "XA.npy"), XA)
    np.save(os.path.join(cache_path, "XB.npy"), XB)
    np.save(os.path.join(cache_path, "y.npy"), y)
    np.savez(os.path.join(cache_path, "meta.npz"), 
             doy=doy, wid=wid, year=year, mask=mask,
             clatA=subA.lat.values, clonA=subA.lon.values,
             clatB=subB.lat.values, clonB=subB.lon.values,
             flat_lat=imd.lat.values, flat_lon=imd.lon.values)
    print("Clean pipeline preparation complete.")

# ======================================================================
# 2. DATASET & CLIMATOLOGY (Readable logic)
# ======================================================================

def get_climatology(data, doys, train_idx, window=7):
    """Computes simple rolling DOY climatology using ONLY training data."""
    train_data = data[train_idx]
    train_doys = doys[train_idx]
    
    # Shape: (366, V, H, W)
    clim = np.zeros((366,) + data.shape[1:], dtype=np.float32)
    for day in range(1, 367):
        # Find indices in training data within the window
        diff = np.abs(train_doys - day)
        in_window = np.minimum(diff, 366 - diff) <= window
        
        if in_window.sum() > 0:
            clim[day-1] = np.nanmean(train_data[in_window], axis=0)
        else:
            # Fallback if window is empty (rare)
            clim[day-1] = np.nanmean(train_data, axis=0)
            
    return clim

class StandardizedDataset(Dataset):
    """
    On-the-fly standardisation. Returns BOTH the anomaly AND the raw 
    climatology so the model knows the physical base state.
    """
    def __init__(self, XA, XB, y, doy, wid, climA, climB, climY, mask, idx):
        self.idx = idx
        self.wid = wid[idx]
        self.mask = torch.from_numpy(mask.astype(np.float32))
        
        # Calculate standard deviations for Z-scoring (using only given idx)
        self.stdA = np.nanstd(XA[idx], axis=(0, 2, 3), keepdims=True)
        self.stdB = np.nanstd(XB[idx], axis=(0, 2, 3), keepdims=True)
        self.stdA = np.where(self.stdA < 1e-6, 1.0, self.stdA)
        self.stdB = np.where(self.stdB < 1e-6, 1.0, self.stdB)
        
        # Pre-process subsets to keep __getitem__ fast
        doys = doy[idx] - 1
        self.XA_anom = (XA[idx] - climA[doys]) / self.stdA
        self.XB_anom = (XB[idx] - climB[doys]) / self.stdB
        
        # We also pass the raw climatology as a feature
        self.XA_clim = climA[doys]
        self.XB_clim = climB[doys]
        
        # Target remains an anomaly in physical units (mm/day)
        self.y_anom = y[idx] - climY[doys]

    def __len__(self): return len(self.idx)
    
    def __getitem__(self, i):
        # Concat anomalies and climatologies so the network sees both
        xa_full = np.concatenate([self.XA_anom[i], self.XA_clim[i]], axis=0)
        xb_full = np.concatenate([self.XB_anom[i], self.XB_clim[i]], axis=0)
        
        return (
            torch.from_numpy(np.nan_to_num(xa_full)),
            torch.from_numpy(np.nan_to_num(xb_full)),
            int(self.wid[i]),
            torch.from_numpy(np.nan_to_num(self.y_anom[i])),
            self.mask
        )

# ======================================================================
# 3. SIMPLIFIED UNET MODEL
# ======================================================================

def gn(c):
    for g in (8, 4, 2, 1):
        if c % g == 0: return nn.GroupNorm(g, c)

class Block(nn.Module):
    def __init__(self, ci, co, drop=0.0):
        super().__init__()
        L = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
             nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
        if drop > 0: L.append(nn.Dropout2d(drop))
        self.f = nn.Sequential(*L)
    def forward(self, x): return self.f(x)

class CleanDualUNet(nn.Module):
    def __init__(self, cA, cB, n_win, base=32, drop=0.3, emb=4):
        super().__init__()
        self.emb = nn.Embedding(n_win, emb)
        
        # Encoders
        self.encA = nn.Sequential(Block(cA + emb, base*2), Block(base*2, base*2, drop))
        self.encB = nn.Sequential(Block(cB, base*2), Block(base*2, base*2, drop))
        
        # Bottleneck (Takes A, B, and static features)
        self.inp = Block(base*4 + 3, base*2)
        self.d1 = Block(base*2, base*4, drop)
        
        # Decoder
        self.u1 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.du1 = Block(base*4, base*2, drop)
        self.head = nn.Conv2d(base*2, 1, 1)

    def forward(self, xa, xb, wid, sampA, sampB, static):
        b = xa.shape[0]
        # Inject Window ID Embedding into Regional block
        e = self.emb(wid)[:, :, None, None].expand(-1, -1, xa.shape[2], xa.shape[3])
        ca = self.encA(torch.cat([xa, e], 1))
        cb = self.encB(xb)
        
        # Align Grids
        fa = F.grid_sample(ca, sampA.expand(b, -1, -1, -1), mode="bilinear", align_corners=True)
        fb = F.grid_sample(cb, sampB.expand(b, -1, -1, -1), mode="bilinear", align_corners=True)
        
        # Concat aligned maps + Lat/Lon/Mask
        f = torch.cat([fa, fb, static.expand(b, -1, -1, -1)], 1)
        H0, W0 = f.shape[-2:]
        f = F.pad(f, (0, (-W0) % 2, 0, (-H0) % 2), mode="replicate")
        
        # U-Net pass
        e0 = self.inp(f)
        e1 = self.d1(F.max_pool2d(e0, 2))
        u = self.du1(torch.cat([self.u1(e1), e0], 1))
        
        return self.head(u)[:, :, :H0, :W0].squeeze(1)

# ======================================================================
# 4. TRAINING LOOP
# ======================================================================

def build_and_run_clean(args, cache_path):
    dev = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    
    # Load raw data straight to RAM (removed mmap since lag bloat is gone)
    print("Loading data to RAM...")
    XA = np.load(os.path.join(cache_path, "XA.npy"))
    XB = np.load(os.path.join(cache_path, "XB.npy"))
    y = np.load(os.path.join(cache_path, "y.npy"))
    z = np.load(os.path.join(cache_path, "meta.npz"))
    
    doy, wid, year, mask = z["doy"], z["wid"], z["year"], z["mask"]
    clatA, clonA = z["clatA"], z["clonA"]
    clatB, clonB = z["clatB"], z["clonB"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    
    # Calculate Channels (Anomalies + Raw Climatology = 2x vars)
    cA = XA.shape[1] * 2 
    cB = XB.shape[1] * 2

    # Pre-build Grid Samplers
    def make_samp(clat, clon):
        gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
        gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
        gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
        return torch.tensor(np.stack([gxx, gyy], -1).astype(np.float32)[None]).to(dev)

    sampA, sampB = make_samp(clatA, clonA), make_samp(clatB, clonB)
    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static = torch.tensor(np.stack([mask, lat2, lon2])[None]).to(dev, dtype=torch.float32)

    # Basic CV Logic
    uy = np.unique(year)
    test_years = set(uy[-TEST_YEARS_N:])
    is_test = np.isin(year, list(test_years))
    nontest_years = sorted(set(uy[~is_test]))
    blocks = np.array_split(nontest_years, args.folds)
    
    for fi, vy_arr in enumerate(blocks):
        vy = set(vy_arr.tolist())
        va_i = np.isin(year, list(vy)) & ~is_test
        tr_i = ~np.isin(year, list(vy)) & ~is_test
        
        tr_idx, va_idx = np.where(tr_i)[0], np.where(va_i)[0]
        print(f"\n--- Fold {fi} | Val Years: {sorted(vy)} ---")
        
        # 1. Compute Climatology strict bounds (ONLY TRAIN)
        with stage("Computing Fold Climatology"):
            climA = get_climatology(XA, doy, tr_idx)
            climB = get_climatology(XB, doy, tr_idx)
            climY = get_climatology(y, doy, tr_idx)
            
        # 2. Build Datasets
        tr_ds = StandardizedDataset(XA, XB, y, doy, wid, climA, climB, climY, mask, tr_idx)
        va_ds = StandardizedDataset(XA, XB, y, doy, wid, climA, climB, climY, mask, va_idx)
        
        tr_dl = DataLoader(tr_ds, batch_size=args.batch, shuffle=True, pin_memory=True)
        va_dl = DataLoader(va_ds, batch_size=args.batch, shuffle=False, pin_memory=True)
        
        # 3. Train
        model = CleanDualUNet(cA, cB, len(WINDOWS), base=args.base, drop=args.drop).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
        
        best, wait = np.inf, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        
        for ep in range(args.epochs):
            model.train()
            tr_loss = 0.0
            for xa, xb, wb, yb, mb in tr_dl:
                xa, xb, wb, yb, mb = [t.to(dev, non_blocking=True) for t in (xa, xb, wb, yb, mb)]
                
                pred = model(xa, xb, wb, sampA, sampB, static)
                # Masked MSE Loss
                loss = ((pred - yb) ** 2 * mb).sum() / mb.sum().clamp(min=1.0)
                
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Stop spikes
                opt.step()
                tr_loss += loss.item()
                
            # Validation
            model.eval()
            vl, n = 0.0, 0
            with torch.no_grad():
                for xa, xb, wb, yb, mb in va_dl:
                    xa, xb, wb, yb, mb = [t.to(dev, non_blocking=True) for t in (xa, xb, wb, yb, mb)]
                    pred = model(xa, xb, wb, sampA, sampB, static)
                    vl += float(((pred - yb) ** 2 * mb).sum() / mb.sum().clamp(min=1.0)) * len(xa)
                    n += len(xa)
            vl /= n
            
            print(f"  Ep {ep:02d} | Tr Loss: {tr_loss/len(tr_dl):.4f} | Va Loss: {vl:.4f}")
            
            if vl < best - 1e-4:
                best, wait = vl, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= args.patience:
                    print(f"  Early stopping triggered.")
                    break
        
        model.load_state_dict(best_state)
        # Note: Implement evaluation logging here...
        break # Stopping at 1 fold for demonstration

if __name__ == "__main__":
    import xarray as xr
    
    # ==========================================
    # 1. LOAD YOUR RAW DATA HERE
    # ==========================================
    # IMPORTANT: Replace these paths with the actual locations of your files!
    print("Loading raw datasets...")
    ds_ecmv = ds_ecmv
    ds_big = ds_big
    ds_imd = ds_imd
    
    if args.cmd == "prepare":
        print(f"Starting data preparation. Caching to: {CACHE_DIR}")
        # This line is now UNCOMMENTED to actually run the caching process:
        prepare_clean(ds_ecmv, ds_big, ds_imd, CACHE_DIR)
        print("Done! Now change args.cmd = 'train' at the top of the file and re-run.")
        
    elif args.cmd == "train":
        if not os.path.exists(os.path.join(CACHE_DIR, "XA.npy")):
            print(f"ERROR: Cache missing. Please set args.cmd = 'prepare' and run first.")
        else:
            print("Starting model training...")
            build_and_run_clean(args, CACHE_DIR)

Loading raw datasets...
Starting data preparation. Caching to: unet_cache_clean


UFuncTypeError: ufunc 'divide' cannot use operands with types dtype('<i8') and dtype('<m8[D]')